In [ ]:
!nvidia-smi

Fri Oct  9 12:20:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os

print(os.listdir("/content"))

['.config', 'mani_sft_validation.jsonl', 'mani_sft_train.jsonl', 'sample_data']


In [ ]:
from huggingface_hub import login
login()

In [ ]:
pip install torch transformers datasets peft trl bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import os
import gc
import torch

# --- PATCH FOR T4 GPU COMPATIBILITY WITH TRL ---
try:
    import trl.kernels.chunked_logprob as cl
    cl._addmm_fp32 = lambda acc, a, b: acc.addmm_(a.float(), b.float())
except Exception:
    pass
# ---------------------------------------------

from huggingface_hub import login
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

# Force clear any lingering GPU memory
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# 1. Configuration (Updated to your mani_project folder)
MODEL_ID = "NCAIR1/N-ATLaS"
DRIVE_DIR = "/content/drive/MyDrive/mani_project"
TRAIN_FILE = os.path.join(DRIVE_DIR, "mani_sft_train.jsonl")
VAL_FILE = os.path.join(DRIVE_DIR, "mani_sft_validation.jsonl")
OUTPUT_DIR = os.path.join(DRIVE_DIR, "natlas_mani_qlora_output")

print("Loading tokenizer and datasets...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

dataset = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "validation": VAL_FILE}
)

def format_dataset(batch):
    formatted_texts = []
    keys = batch.keys()

    c_key = next((k for k in ['context', 'input', 'prompt'] if k in keys), None)
    q_key = next((k for k in ['question', 'instruction'] if k in keys), None)
    a_key = next((k for k in ['reference_answer', 'output', 'answer', 'completion'] if k in keys), None)

    col_list = list(keys)

    for i in range(len(batch[col_list[0]])):
        c_val = batch[c_key][i] if c_key else ""
        q_val = batch[q_key][i] if q_key else batch[col_list[0]][i]
        a_val = batch[a_key][i] if a_key else batch[col_list[1]][i]

        if c_val:
            text = f"Context: {c_val}\n\nQuestion: {q_val}\n\nAnswer: {a_val}{tokenizer.eos_token}"
        else:
            text = f"Question: {q_val}\n\nAnswer: {a_val}{tokenizer.eos_token}"

        formatted_texts.append(text)

    return {"text": formatted_texts}

print("Formatting dataset splits...")
dataset["train"] = dataset["train"].map(format_dataset, batched=True, remove_columns=dataset["train"].column_names)
dataset["validation"] = dataset["validation"].map(format_dataset, batched=True, remove_columns=dataset["validation"].column_names)

# 2. Configure 4-bit Quantization for T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading base model {MODEL_ID} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="cuda:0",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

model.gradient_checkpointing_enable()

# 3. Configure LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 4. Training Arguments with frequent Google Drive backups
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=20,
    save_steps=20,
    fp16=False,
    bf16=False,
    num_train_epochs=3,
    max_grad_norm=0.3,
    warmup_steps=10,
    save_total_limit=2,
    report_to="none",
    dataset_text_field="text",
    max_length=1024,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    args=training_args,
)

checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    checkpoints = [os.path.join(OUTPUT_DIR, d) for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")]
    if checkpoints:
        checkpoint = max(checkpoints, key=os.path.getmtime)
        print(f"--> Found existing checkpoint! Resuming training from: {checkpoint}")

print("Starting QLoRA Fine-Tuning...")
trainer.train(resume_from_checkpoint=checkpoint)

print(f"Saving final fine-tuned adapter weights to {OUTPUT_DIR}...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Training completed successfully and saved to Google Drive!")

Loading tokenizer and datasets...


config.json:   0%|          | 0.00/875 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Dropping fully masked examples from eval dataset:   0%|          | 0/360 [00:00<?, ? examples/s]

Starting QLoRA Fine-Tuning...


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
20,0.737744,0.636526,0.628124,0.860632,39692.000000
40,0.587011,0.582445,0.571429,0.868043,79327.000000
60,0.565989,0.570068,0.546264,0.869883,118881.000000
80,0.557770,0.565982,0.545679,0.870507,158345.000000
100,0.544382,0.557663,0.536045,0.872032,198063.000000
120,0.539838,0.553887,0.538679,0.871913,237659.000000
140,0.536603,0.551641,0.530571,0.872817,277328.000000
160,0.525394,0.543706,0.533311,0.873675,316601.000000
180,0.515249,0.541177,0.523316,0.874881,356097.000000
200,0.497342,0.539620,0.502208,0.875293,395912.000000


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
20,0.737744,0.636526,0.628124,0.860632,39692.000000
40,0.587011,0.582445,0.571429,0.868043,79327.000000
60,0.565989,0.570068,0.546264,0.869883,118881.000000
80,0.557770,0.565982,0.545679,0.870507,158345.000000
100,0.544382,0.557663,0.536045,0.872032,198063.000000
120,0.539838,0.553887,0.538679,0.871913,237659.000000
140,0.536603,0.551641,0.530571,0.872817,277328.000000
160,0.525394,0.543706,0.533311,0.873675,316601.000000
180,0.515249,0.541177,0.523316,0.874881,356097.000000
200,0.497342,0.539620,0.502208,0.875293,395912.000000


Saving final fine-tuned adapter weights to /content/drive/MyDrive/mani_project/natlas_mani_qlora_output...
Training completed successfully and saved to Google Drive!


In [10]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# 1. Clear out any lingering VRAM from the training session
torch.cuda.empty_cache()
gc.collect()

ADAPTER_PATH = "/content/drive/MyDrive/mani_project/natlas_mani_qlora_output"
BASE_MODEL_ID = "NCAIR1/N-ATLaS"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)

# 2. Configure 4-bit quantization to prevent OOM errors on the T4 GPU
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print("Loading base model in 4-bit quantization...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)

print("Loading fine-tuned MANI LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

# Sample public health test prompt blending CHIRPS, NDVI, and DHS biomarker inputs
test_prompt = (
    "Context: Satellite telemetry indicates a 35% increase in CHIRPS precipitation anomalies and a rising NDVI vector habitat index across LGA North. "
    "DHS biomarker survey logs report an RDT parasitemia prevalence of 28% and low household ITN (insecticide-treated net) usage rates.\n\n"
    "Question: Based on these epidemiological indicators, what immediate targeted resource allocation and intervention should the state health manager prioritize for this zone?\n\n"
    "Answer:"
)

print("\n--- Generating MANI Response ---")
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(decoded_output)

Loading tokenizer...
Loading base model in 4-bit quantization...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading fine-tuned MANI LoRA adapter...

--- Generating MANI Response ---


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Context: Satellite telemetry indicates a 35% increase in CHIRPS precipitation anomalies and a rising NDVI vector habitat index across LGA North. DHS biomarker survey logs report an RDT parasitemia prevalence of 28% and low household ITN (insecticide-treated net) usage rates.

Question: Based on these epidemiological indicators, what immediate targeted resource allocation and intervention should the state health manager prioritize for this zone?

Answer: LGA North should prioritize surveillance, RDTs, rainfall interpretation, and environmental intervention.
